# 04 - Conversation Memory

This notebook tests the assistant with memory (`build_chain()`). I check three things:

1. **Memory works:** it remembers what the user said earlier in the same session, such as their name.
2. **Follow-up questions:** the retriever only sees the current message, so a question like "how much does it cost?" may not find the right FAQ.
3. **Sessions are separate:** a new session shouldn't know anything from another one.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.rag_chain import build_chain, build_retriever, load_vectorstore
from dotenv import load_dotenv

load_dotenv()

chain = build_chain()
retriever = build_retriever(load_vectorstore())

/var/folders/w_/gkk9ld415875n69t0tn3s4f40000gn/T/ipykernel_52091/2652350238.py:12: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = build_chain()


## Test 1: a three-turn conversation

The same conversation as my original script. All three messages use the same session id, so they share one history.

In [2]:
session = {"configurable": {"session_id": "lemuel"}}

messages = [
    "Hi, I'm Lemuel. What is this Delivery Club all about?",
    "Ok great, can you confirm how much it costs please",
    "What is my name?",
]

for message in messages:
    passed_ids = [doc.metadata["id"] for doc in retriever.invoke(message)]
    response = chain.invoke({"input": message}, config=session)

    print("USER:      ", message)
    print("RETRIEVED: ", passed_ids)
    print("ASSISTANT: ", response.content)
    print()

/Users/lemuel/Documents/de/project/data-engineering/rag-gen-ai-qa-bot/notebooks/../src/rag_chain.py:120: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  _session_store[session_id] = ChatMessageHistory()


USER:       Hi, I'm Lemuel. What is this Delivery Club all about?
RETRIEVED:  ['0005', '0006', '0007', '0001']
ASSISTANT:  Hi Lemuel! Delivery Club is ABC Grocery’s unlimited delivery membership. For $100 per year, you get free delivery on eligible orders. Without membership, delivery is $10 per order. It doesn’t change product prices or tips—it just removes the per‑delivery fee. The $35 minimum order subtotal for delivery still applies.



No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33


USER:       Ok great, can you confirm how much it costs please
RETRIEVED:  []
ASSISTANT:  I don’t have that information in the provided context. Please email human@abc-grocery.com and they will be glad to assist you!



No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33


USER:       What is my name?
RETRIEVED:  []
ASSISTANT:  Your name is Lemuel.



- **Turn 1:** retrieval found 0005 (Delivery Club) first, and the assistant used my name from the message. Every fact in the answer comes from the retrieved FAQs.
- **Turn 2 (follow-up):** the retriever only sees the current message. "How much it costs" doesn't mention Delivery Club, so no chunk passed the threshold and the assistant gave the fallback, even though it had said the price ($100) one turn earlier. This follows rule 3 (history can't be used for company facts), but it's a poor result for the customer.
- **Turn 3:** the context was empty, but the assistant answered from history. A name isn't company information, so rule 6 (personalisation) applies rather than the fallback rule.

The "No relevant docs" warnings appear twice for each message because the retriever runs twice: once for the `RETRIEVED` line and once inside the chain.

## Test 2: a new session

A different session id should start with an empty history, so the assistant shouldn't know Andrew's name.

In [3]:
new_session = {"configurable": {"session_id": "John"}}

response = chain.invoke({"input": "What is my name?"}, config=new_session)
print(response.content)

No relevant docs were retrieved using the relevance score threshold 0.33


I don’t know your name yet. What should I call you?


A new session doesn't know the name, so each session has its own history and nothing leaks between users.

## Summary

- **Memory works within a session:** the assistant remembered my name and used it to personalise answers.
- **Sessions are separate:** a new session started with no history.
- **Follow-up questions fail.** The retriever only sees the current message, so "how much does it cost?" retrieved nothing and the assistant gave the fallback. The grounding rules stop it from using the earlier answer in its history, so nothing wrong was said, but the customer didn't get an answer.
- **The fix:** before retrieving, rewrite the follow-up into a standalone question using the history ("How much does Delivery Club cost?"), then search with that. This is a common step in conversational RAG, and it's the next improvement for this project.
- Memory is kept in a Python dictionary, so it's lost when the program stops. In production it could be stored in something like Redis or a database.